In [2]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


# Practice 2 — Challenge A: Cybersecurity

**Machine Learning — 2026/2027**

This notebook contains the Challenge A

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
    cross_validate,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

pd.set_option("display.max_columns", None)

PROJECT_DIR = Path(".")
CYBER_DIR = PROJECT_DIR / "datos" / "cyber"

CYBER_DEVELOPMENT_FILE = CYBER_DIR / "cyber_development.csv"
CYBER_GROUPS_FILE = CYBER_DIR / "cyber_groups.csv"
CYBER_CLASS_CODES_FILE = CYBER_DIR / "cyber_class_codes.json"
CYBER_CANONICAL_FOLDS_FILE = CYBER_DIR / "cyber_folds.csv"

## P2.1 — Data contract and exploration

The statement requires us to:

1. load the development table and verify its schema;
2. report the support of every class, naming ransomware explicitly;
3. report the number of distinct `feature_group` values.

`feature_group` is grouping metadata. It is used to define the evaluation folds and is **not** an estimator feature.

In [5]:
cyber = pd.read_csv(CYBER_DEVELOPMENT_FILE)
cyber_groups = pd.read_csv(CYBER_GROUPS_FILE)

with open(CYBER_CLASS_CODES_FILE, "r", encoding="utf-8") as f:
    class_codes = {int(code): name for code, name in json.load(f).items()}

NOMINAL_FEATURES = ["proto", "service", "conn_state"]
NUMERIC_FEATURES = [
    "duration",
    "src_bytes",
    "dst_bytes",
    "missed_bytes",
    "src_pkts",
    "src_ip_bytes",
    "dst_pkts",
    "dst_ip_bytes",
]
MODEL_FEATURES = NOMINAL_FEATURES + NUMERIC_FEATURES

EXPECTED_DEVELOPMENT_COLUMNS = ["row_id"] + MODEL_FEATURES + ["attack_code"]
EXPECTED_GROUP_COLUMNS = ["row_id", "feature_group"]

assert list(cyber.columns) == EXPECTED_DEVELOPMENT_COLUMNS
assert list(cyber_groups.columns) == EXPECTED_GROUP_COLUMNS
assert cyber["row_id"].is_unique
assert cyber_groups["row_id"].is_unique
assert set(cyber["row_id"]) == set(cyber_groups["row_id"])

print("Development shape:", cyber.shape)
print("Development columns:")
print(list(cyber.columns))
print("\nGroup metadata shape:", cyber_groups.shape)
print("Group metadata columns:")
print(list(cyber_groups.columns))

Development shape: (34928, 13)
Development columns:
['row_id', 'proto', 'service', 'conn_state', 'duration', 'src_bytes', 'dst_bytes', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'attack_code']

Group metadata shape: (34928, 2)
Group metadata columns:
['row_id', 'feature_group']


In [4]:
class_support = (
    cyber["attack_code"]
    .value_counts()
    .sort_index()
    .rename("support")
    .to_frame()
)

class_support["class_name"] = class_support.index.map(class_codes)
class_support = class_support[["class_name", "support"]]

display(class_support)

ransomware_code = next(
    code for code, name in class_codes.items()
    if name == "ransomware"
)
ransomware_support = int(class_support.loc[ransomware_code, "support"])

n_feature_groups = cyber_groups["feature_group"].nunique()

print("Ransomware support:", ransomware_support)
print("Distinct feature_group values:", n_feature_groups)

,class_name,support
attack_code,,
0,backdoor,1573
1,ddos,4800
2,dos,4800
3,injection,4800
4,mitm,805
5,normal,4800
6,password,4800
7,ransomware,208
8,scanning,3542


Ransomware support: 208
Distinct feature_group values: 34883


### P2.1 result

The cybersecurity development set contains 34,928 observations and 11 model features. The ten class supports are reported above. Ransomware is the rarest class, with a support of 208 observations. The grouping metadata contains 34,883 distinct feature_group values, confirming that some observations share an exact feature group.

## P2.2(a) — Construct the grouped five-fold protocol

For Challenge A the required development protocol is a five-fold `StratifiedGroupKFold`:

- stratification uses `attack_code`;
- grouping uses `feature_group`;
- the same exact feature group must never appear in both training and validation within a fold.

The Unit 5 notebook uses a five-fold stratified splitter shuffled once with the fixed seed. We follow the same course pattern here, adding the group constraint required by the Practice 2 statement.

In [5]:
cyber_all = cyber.merge(
    cyber_groups,
    on="row_id",
    how="left",
    validate="one_to_one",
)

X_cyber = cyber_all[MODEL_FEATURES]
y_cyber = cyber_all["attack_code"]
groups_cyber = cyber_all["feature_group"]

cv_grouped = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=PROJECT_RANDOM_STATE,
)

own_fold_id = np.empty(len(cyber_all), dtype=int)
fold_sizes = []

for q, (train_idx, valid_idx) in enumerate(
    cv_grouped.split(X_cyber, y_cyber, groups=groups_cyber),
    start=1,
):
    own_fold_id[valid_idx] = q
    fold_sizes.append({
        "fold": q,
        "train_rows": len(train_idx),
        "validation_rows": len(valid_idx),
    })

own_cyber_folds = pd.DataFrame({
    "row_id": cyber_all["row_id"],
    "fold_id": own_fold_id,
})

display(pd.DataFrame(fold_sizes))
display(own_cyber_folds.head())

,fold,train_rows,validation_rows
0,1,27942,6986
1,2,27942,6986
2,3,27943,6985
3,4,27942,6986
4,5,27943,6985


,row_id,fold_id
0,C00000,2
1,C00001,1
2,C00002,3
3,C00003,2
4,C00004,1


## P2.2(b) — Verify the grouping invariant

The invariant for Challenge A is:

> no exact `feature_group` appears in both the training and validation portions of any fold.

We verify that directly for all five grouped folds. We also inspect an ordinary row-wise `StratifiedKFold` to show what changes when the grouping constraint is removed.

In [6]:
grouped_invariant_rows = []

for q, (train_idx, valid_idx) in enumerate(
    cv_grouped.split(X_cyber, y_cyber, groups=groups_cyber),
    start=1,
):
    train_groups = set(groups_cyber.iloc[train_idx])
    valid_groups = set(groups_cyber.iloc[valid_idx])
    overlap = train_groups.intersection(valid_groups)

    grouped_invariant_rows.append({
        "fold": q,
        "shared_feature_groups": len(overlap),
        "invariant_holds": len(overlap) == 0,
    })

grouped_invariant_report = pd.DataFrame(grouped_invariant_rows)
display(grouped_invariant_report)

assert grouped_invariant_report["invariant_holds"].all()

,fold,shared_feature_groups,invariant_holds
0,1,0,True
1,2,0,True
2,3,0,True
3,4,0,True
4,5,0,True


In [7]:
cv_rowwise = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=PROJECT_RANDOM_STATE,
)

rowwise_overlap_rows = []

for q, (train_idx, valid_idx) in enumerate(
    cv_rowwise.split(X_cyber, y_cyber),
    start=1,
):
    train_groups = set(groups_cyber.iloc[train_idx])
    valid_groups = set(groups_cyber.iloc[valid_idx])
    overlap = train_groups.intersection(valid_groups)

    rowwise_overlap_rows.append({
        "fold": q,
        "shared_feature_groups": len(overlap),
    })

rowwise_overlap_report = pd.DataFrame(rowwise_overlap_rows)
display(rowwise_overlap_report)

,fold,shared_feature_groups
0,1,13
1,2,13
2,3,14
3,4,10
4,5,15


### P2.2(b) interpretation

The grouped protocol satisfies the required invariant in every fold: the number of shared `feature_group` values is zero.

Ordinary row-wise stratified cross-validation does not know about `feature_group`. Therefore, rows belonging to the same exact feature group can be placed on opposite sides of a fold. The table above shows that this occurs in the row-wise split.

## P2.2(d) — Grouped versus row-wise evaluation on Challenge A

The Practice 2 statement fixes the comparison pipeline so that the **only** thing that changes between the two runs is the splitting protocol.

For Challenge A:

- estimator: `KNeighborsClassifier(n_neighbors=5)`;
- nominal predictors: one-hot encoding;
- numeric predictors: `log1p` followed by standardisation;
- fitted preprocessing stays inside the fold;
- metric: macro F1.

The grouped run uses `StratifiedGroupKFold`.  
The row-wise run uses `StratifiedKFold`.

In [8]:
numeric_preprocessing = Pipeline([
    ("log1p", FunctionTransformer(np.log1p)),
    ("scale", StandardScaler()),
])

preprocessing = ColumnTransformer([
    ("nominal", OneHotEncoder(handle_unknown="ignore"), NOMINAL_FEATURES),
    ("numeric", numeric_preprocessing, NUMERIC_FEATURES),
])

cyber_comparison_pipeline = Pipeline([
    ("preprocess", preprocessing),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

grouped_scores = cross_validate(
    cyber_comparison_pipeline,
    X_cyber,
    y_cyber,
    groups=groups_cyber,
    cv=cv_grouped,
    scoring="f1_macro",
    n_jobs=1,
)["test_score"]

rowwise_scores = cross_validate(
    cyber_comparison_pipeline,
    X_cyber,
    y_cyber,
    cv=cv_rowwise,
    scoring="f1_macro",
    n_jobs=1,
)["test_score"]

comparison_scores = pd.DataFrame({
    "fold": np.arange(1, 6),
    "grouped_macro_f1": grouped_scores,
    "rowwise_macro_f1": rowwise_scores,
})

display(comparison_scores.round(4))

grouped_mean = grouped_scores.mean()
rowwise_mean = rowwise_scores.mean()
observed_difference = rowwise_mean - grouped_mean

print(f"Grouped mean macro F1 : {grouped_mean:.4f}")
print(f"Row-wise mean macro F1: {rowwise_mean:.4f}")
print(f"Row-wise - grouped     : {observed_difference:+.4f}")

,fold,grouped_macro_f1,rowwise_macro_f1
0,1,0.9419,0.9372
1,2,0.9466,0.9404
2,3,0.9516,0.9476
3,4,0.9399,0.9536
4,5,0.9408,0.9424


Grouped mean macro F1 : 0.9442
Row-wise mean macro F1: 0.9443
Row-wise - grouped     : +0.0001


### P2.2(d) interpretation

The observed row-wise minus grouped difference is the value printed above.

In this run, the two mean macro-F1 values are almost identical. The row-wise estimate is only slightly higher. The Practice 2 statement explicitly says that, for Challenge A, the sign and size of this difference are empirical results and are not predicted in advance.

## P2.2(e) — Why splitting identical feature groups is methodologically undesirable

The size of the observed gap does not determine whether grouping is methodologically necessary.

If the same exact `feature_group` is allowed to appear in both training and validation, the validation score no longer measures generalisation to a **different exact feature group**. The grouped protocol prevents that contamination by keeping every exact feature group entirely on one side of each fold.

## P2.2(f) — Freeze our construction and compare with the canonical folds

At this step the statement says that the course-owned `cyber_folds.csv` is released.

The initial project package does not contain that file, so this section is written to perform the comparison as soon as the canonical file is placed in `datos/cyber/`.

Our own fold construction and invariant report above are now frozen and should not be rewritten after seeing the canonical assignment.

In [9]:
if CYBER_CANONICAL_FOLDS_FILE.exists():
    canonical_folds = pd.read_csv(CYBER_CANONICAL_FOLDS_FILE)

    assert list(canonical_folds.columns) == ["row_id", "fold_id"]
    assert canonical_folds["row_id"].is_unique
    assert set(canonical_folds["row_id"]) == set(own_cyber_folds["row_id"])

    fold_comparison = own_cyber_folds.merge(
        canonical_folds,
        on="row_id",
        how="inner",
        suffixes=("_ours", "_canonical"),
        validate="one_to_one",
    )

    fold_comparison["same_fold_id"] = (
        fold_comparison["fold_id_ours"]
        == fold_comparison["fold_id_canonical"]
    )

    print(
        "Rows with the same fold id:",
        int(fold_comparison["same_fold_id"].sum()),
        "/",
        len(fold_comparison),
    )
    print(
        "Exact fold-id agreement:",
        f"{fold_comparison['same_fold_id'].mean():.4f}",
    )

    display(pd.crosstab(
        fold_comparison["fold_id_ours"],
        fold_comparison["fold_id_canonical"],
        rownames=["our fold"],
        colnames=["canonical fold"],
    ))
else:
    print(
        "P2.2(f) cannot be completed yet: "
        "datos/cyber/cyber_folds.csv is not present in the initial package."
    )

P2.2(f) cannot be completed yet: datos/cyber/cyber_folds.csv is not present in the initial package.


### P2.2(f) interpretation

**To complete when the course-owned file is released.**

Compare the frozen team assignment with the canonical assignment and describe the discrepancy shown by the table above. From P2.2(g) onward, the canonical fold assignment becomes the project authority.